In [1]:
import pandas as pd
import numpy as np

# ================================================================= #
# PART A, STEP 1: LOAD DATASETS & DOCUMENT                          #
# ================================================================= #

trader_df = pd.read_csv("historical_data.csv")
sentiment_df = pd.read_csv("fear_greed_index.csv")

# ================================================================= #
# CALCULATE SHAPE, MISSING VALUES, AND DUPLICATES                   #
# ================================================================= #

print("--- Trader Data ---")
print(f"Rows: {trader_df.shape[0]}, Columns: {trader_df.shape[1]}")
print(f"Missing Values: {trader_df.isnull().sum().sum()}")
print(f"Duplicates: {trader_df.duplicated().sum()}")

print("\n--- Sentiment Data ---")
print(f"Rows: {sentiment_df.shape[0]}, Columns: {sentiment_df.shape[1]}")
print(f"Missing Values: {sentiment_df.isnull().sum().sum()}")
print(f"Duplicates: {sentiment_df.duplicated().sum()}")

--- Trader Data ---
Rows: 46482, Columns: 16
Missing Values: 5
Duplicates: 0

--- Sentiment Data ---
Rows: 2644, Columns: 4
Missing Values: 0
Duplicates: 0


/tmp/ipykernel_4613/2324808887.py:8: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  trader_df = pd.read_csv("historical_data.csv")


In [2]:
# ================================================================= #
# PART A, STEP 2: CONVERT TIMESTAMPS & ALIGN DATASETS               #
# ================================================================= #

# 1. Clean the 5 missing values found in the previous step
trader_df.dropna(inplace=True)

# 2. Convert timestamps to a standard 'YYYY-MM-DD' daily string
# Using errors='coerce' and dayfirst=True prevents formatting crashes
trader_df['Date'] = pd.to_datetime(trader_df['Timestamp IST'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')
sentiment_df['Date'] = pd.to_datetime(sentiment_df['date']).dt.strftime('%Y-%m-%d')

# 3. Merge datasets based on the Date (Left Join)
merged_df = pd.merge(trader_df, sentiment_df[['Date', 'classification', 'value']], on='Date', how='left')

# Drop any outlier trades where sentiment data wasn't available for that day
merged_df.dropna(subset=['classification'], inplace=True)
print(f"✅ Merged Dataset Ready! Shape: {merged_df.shape}")

# ================================================================= #
# PART A, STEP 3: CREATE KEY METRICS                                #
# ================================================================= #

# Helper column: 1 if it's a winning trade, 0 if it's a losing trade
merged_df['Is_Win'] = (merged_df['Closed PnL'] > 0).astype(int)

# Group the data to calculate daily PnL, win rate, and average trade size per account
daily_metrics = merged_df.groupby(['Account', 'Date']).agg(
    Daily_PnL=('Closed PnL', 'sum'),
    Total_Trades=('Closed PnL', 'count'),
    Winning_Trades=('Is_Win', 'sum'),
    Avg_Trade_Size_USD=('Size USD', 'mean')
).reset_index()

# Calculate the Win Rate decimal
daily_metrics['Win_Rate'] = daily_metrics['Winning_Trades'] / daily_metrics['Total_Trades']

print("✅ Daily Metrics Successfully Engineered!")
daily_metrics.head(3)

✅ Merged Dataset Ready! Shape: (46475, 19)
✅ Daily Metrics Successfully Engineered!


,Account,Date,Daily_PnL,Total_Trades,Winning_Trades,Avg_Trade_Size_USD,Win_Rate
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,0.0,177,0,5089.718249,0.0
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,0.0,68,0,7976.664412,0.0
2,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-18,0.0,40,0,23734.500000,0.0


In [3]:
# @title
import plotly.express as px

# ================================================================= #
# PART B: DATA ANALYSIS & INSIGHTS                                  #
# ================================================================= #

print("--- 1. Performance by Market Sentiment ---")
# Calculate Win Rate and Average PnL for each Sentiment Classification
performance = merged_df.groupby('classification').agg(
    Total_Trades=('Closed PnL', 'count'),
    Avg_PnL=('Closed PnL', 'mean'),
    Win_Rate=('Is_Win', lambda x: x.sum() / len(x))
).reset_index()
print(performance.to_string(index=False))

print("\n--- 2. Trader Behavior by Market Sentiment ---")
# Calculate Average Position Size and what % of trades are BUY (Long Ratio)
behavior = merged_df.groupby('classification').agg(
    Avg_Trade_Size_USD=('Size USD', 'mean'),
    Long_Ratio=('Side', lambda x: (x == 'BUY').sum() / len(x))
).reset_index()
print(behavior.to_string(index=False))

print("\n--- 3. Trader Segmentation (Frequent vs Infrequent) ---")
# Classify traders with >100 trades as 'Frequent', else 'Infrequent'
trade_counts = merged_df['Account'].value_counts()
freq_traders = trade_counts[trade_counts > 100].index
merged_df['Segment'] = np.where(merged_df['Account'].isin(freq_traders), 'Frequent', 'Infrequent')

# Check how segments perform during different market sentiments
segment_perf = merged_df.groupby(['Segment', 'classification'])['Closed PnL'].mean().reset_index()
print(segment_perf.head()) # Previewing the segment table

# ================================================================= #
# PART B: VISUALIZATIONS (PLOTLY)                                   #
# ================================================================= #

# Chart 1: Average PnL by Sentiment
fig1 = px.bar(
    performance,
    x='classification',
    y='Avg_PnL',
    title="Average PnL per Trade by Market Sentiment",
    color='classification',
    text_auto='.2f'
)
fig1.show()

# Chart 2: Buy (Long) Ratio by Sentiment
fig2 = px.bar(
    behavior,
    x='classification',
    y='Long_Ratio',
    title="Long (BUY) Bias by Market Sentiment",
    color='classification',
    text_auto='.1%'
)
fig2.show()

--- 1. Performance by Market Sentiment ---
classification  Total_Trades    Avg_PnL  Win_Rate
  Extreme Fear          1746 204.336799  0.404926
 Extreme Greed          9354 162.759181  0.536883
          Fear         12114 146.672879  0.445022
         Greed         15166  92.228518  0.411315
       Neutral          8095  86.191512  0.479061

--- 2. Trader Behavior by Market Sentiment ---
classification  Avg_Trade_Size_USD  Long_Ratio
  Extreme Fear         9791.737211    0.569301
 Extreme Greed         8168.774757    0.452641
          Fear        23119.389312    0.528314
         Greed        14226.886005    0.483780
       Neutral        13987.707452    0.508833

--- 3. Trader Segmentation (Frequent vs Infrequent) ---
    Segment classification  Closed PnL
0  Frequent   Extreme Fear  204.336799
1  Frequent  Extreme Greed  162.759181
2  Frequent           Fear  146.672879
3  Frequent          Greed   92.228518
4  Frequent        Neutral   86.191512
